# VALIDATION COPY — internal only

Identical to the release notebook except that it reads `data_full.csv`
(all 20 years) and the last cell produces and scores the submission.

**Do not distribute.**


---
## 0 · Setup

In [ ]:
# DO NOT TOUCH
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.stattools import adfuller

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# ---- configuration -------------------------------------------------
DATA_CSV = "data_full.csv"

TRAIN_END    = pd.Timestamp("2019-12-31")   # last day of the training period
MAX_FEATURES = 5                            # per target
TRADING_DAYS = 252

---
## 1 · Loading

Reads the CSV and nothing else — no files are written, no cache is created.

`data.csv` has one row per business day and seventeen columns:

| column | |
|---|---|
| `IDX` | closing level of the market index |
| `e1` … `e10` | closing prices of ten stocks |
| `ETF` | closing price of an exchange-traded fund |
| `OPT_K` | strike of the currently listed option contract |
| `OPT_T` | business days remaining to its expiry |
| `OPT_CALL`, `OPT_PUT` | its call and put prices |
| `OPT_IV` | its implied volatility, annualised |

The option contracts are quarterly and are struck at-the-money when written. Their
underlying is `e2`. There are no missing values and no duplicated dates.

In [ ]:
# DO NOT TOUCH
def load_data(path=DATA_CSV):
    """Read the data file. Returns a DataFrame indexed by date."""
    return pd.read_csv(path, parse_dates=["Date"]).set_index("Date").sort_index()

---
## 2 · Panel construction

`build_panel` turns the raw file into every representation used downstream: daily
returns, weekly aggregates, and weekly realized volatility. The weekly grid is Friday-ended.

**Definitions.** Let $P_d$ be a closing price and $r_d = \log(P_d / P_{d-1})$ the daily log
return. For week $w$, containing $n_w$ trading days:

$$r_w \;=\; \sum_{d \in w} r_d
\qquad\qquad
RV_w \;=\; \sqrt{\frac{252}{n_w} \sum_{d \in w} r_d^{\,2}}$$

$r_w$ is `panel.wret`; $RV_w$ is `panel.wrv`, annualised. Weeks with fewer than three
trading days are dropped from the weekly grid.

**Alignment convention — read this once, then you can forget about it.**
Every weekly series is indexed by the Friday that *ends* the week. Row `t` of a feature may
only contain information available on Friday `t`. Row `t` of a target holds the value realised
over the *following* week. So features and targets on the same row are already correctly
lined up, and you never need to shift anything yourself.

In [ ]:
# DO NOT TOUCH
class Panel:
    """Container for every derived representation of the raw data.

    Attributes
    ----------
    px          daily prices (IDX, e1..e10, ETF)
    ret         daily log returns
    opt         daily option data
    wpx         weekly prices, last observation of each week
    wret        weekly log returns, summed within each week
    wopt        weekly option data, last observation of each week
    wrv         weekly realized volatility, annualised
    index       the weekly Friday DatetimeIndex shared by all weekly frames
    names       the ten stock tickers
    """

    def __init__(self, px, ret, opt, wpx, wret, wopt, wrv):
        self.px, self.ret, self.opt = px, ret, opt
        self.wpx, self.wret, self.wopt, self.wrv = wpx, wret, wopt, wrv
        self.index = wret.index
        self.names = [c for c in px.columns if c.startswith("e")]

    def __repr__(self):
        return (f"<Panel {len(self.px)} daily rows "
                f"({self.px.index.min().date()} to {self.px.index.max().date()}), "
                f"{len(self.index)} weekly rows>")


def to_returns(prices):
    """Daily log returns of a price frame."""
    return np.log(prices).diff()


def to_weekly(daily, how="sum"):
    """Aggregate a daily frame onto a Friday-ended weekly grid.
    how: 'sum' | 'last' | 'mean' | 'count'."""
    g = daily.resample("W-FRI")
    if   how == "sum":   return g.sum(min_count=1)
    elif how == "last":  return g.last()
    elif how == "mean":  return g.mean()
    elif how == "count": return g.count()
    raise ValueError(f"unknown aggregation: {how}")

def build_panel(data=None, min_days_per_week=3):
    """Build the full Panel from the raw file. Columns whose name starts with 'OPT_' are
    treated as option data, the rest as instrument prices. Weeks with fewer than
    `min_days_per_week` observations are dropped from the weekly grid."""
    df  = load_data() if data is None else data
    opt = df[[c for c in df.columns if c.startswith("OPT_")]]
    px  = df[[c for c in df.columns if not c.startswith("OPT_")]]
    ret = to_returns(px)

    keep = to_weekly(ret[["IDX"]], "count")["IDX"] >= min_days_per_week

    wret = to_weekly(ret,  "sum")[keep]
    wpx  = to_weekly(px,   "last")[keep]
    wopt = to_weekly(opt,  "last")[keep]

    sq = (ret ** 2).resample("W-FRI")
    wrv = np.sqrt(sq.sum(min_count=1) * TRADING_DAYS / sq.count())[keep]

    return Panel(px, ret, opt, wpx, wret, wopt, wrv)

In [ ]:
# DO NOT TOUCH
panel = build_panel()
panel

In [ ]:
# DO NOT TOUCH
# A look at each representation the Panel carries.
print("tickers     :", panel.names)
print("daily rows  :", len(panel.px))
print("weekly rows :", len(panel.index),
      f"  ({panel.index.min().date()} -> {panel.index.max().date()})")

display(panel.px.tail(3))     # daily prices
display(panel.ret.tail(3))    # daily log returns
display(panel.wret.tail(3))   # weekly log returns          r_w
display(panel.wrv.tail(3))    # weekly realized volatility  RV_w, annualised
display(panel.wopt.tail(3))   # weekly option data, last observation of each week

---
## 3 · Targets and the train / holdout split

Each target is a weekly series aligned to the convention above: the value on row `t` is what
happens during the week *after* Friday `t`.

$$T1_t = r^{e1}_{t+1} \qquad T2_t = RV^{e2}_{t+1} \qquad T3_t = r^{ETF}_{t+1}$$

In [ ]:
# DO NOT TOUCH
def make_target_t1(panel):
    """T1 — log return of e1 over the following week."""
    return panel.wret["e1"].shift(-1).rename("T1")


def make_target_t2(panel):
    """T2 — annualised realized volatility of e2 over the following week."""
    return panel.wrv["e2"].shift(-1).rename("T2")


def make_target_t3(panel):
    """T3 — log return of the ETF over the following week."""
    return panel.wret["ETF"].shift(-1).rename("T3")


def split_masks(panel, train_end=TRAIN_END):
    """Boolean masks over the weekly index.

    A row belongs to `train` when both the prediction date and the week being predicted
    fall inside the training period; to `holdout` when the prediction date is after it.
    The single week straddling the boundary belongs to neither, and neither does the final
    row of the grid, whose following week is not in the data."""
    idx     = panel.index
    nxt     = pd.Series(idx, index=idx).shift(-1)
    train   = (idx <= train_end) & (nxt <= train_end)
    holdout = (idx > train_end) & nxt.notna().values
    return (pd.Series(train, index=idx, name="train"),
            pd.Series(holdout, index=idx, name="holdout"))

In [ ]:
# DO NOT TOUCH
targets = {"T1": make_target_t1(panel),
           "T2": make_target_t2(panel),
           "T3": make_target_t3(panel)}
IS_TRAIN, IS_HOLDOUT = split_masks(panel)


def _span(mask):
    idx = panel.index[mask]
    return "none in this copy of the data" if len(idx) == 0 else \
           f"{idx.min().date()} -> {idx.max().date()}"


print(f"train weeks  : {IS_TRAIN.sum():>4}   {_span(IS_TRAIN)}")
print(f"scored weeks : {IS_HOLDOUT.sum():>4}   {_span(IS_HOLDOUT)}")
pd.DataFrame(targets).describe().T

---
## 4 · Toolbox

A collection of general-purpose analysis functions.

**Not all of them are useful for this problem.** Some are here because they are the sort of
thing one reaches for when looking at financial data, not because they help. Deciding what to
ignore is part of the exercise. You are also free to write your own.

In [ ]:
# DO NOT TOUCH -- toolbox
# ---------------------------------------------------------------- regression

def ols(y, X, add_const=True):
    """Least-squares fit of y on X. Returns (coefficients, fitted, residuals).
    X may be a Series, DataFrame or ndarray. Rows with missing values are dropped
    from the fit; fitted values and residuals are returned on the original index."""
    y = pd.Series(y)
    X = pd.DataFrame(X)
    if add_const:
        X = X.assign(const=1.0)
    ok = y.notna() & X.notna().all(axis=1)
    b, *_ = np.linalg.lstsq(X[ok].values, y[ok].values, rcond=None)
    b = pd.Series(b, index=X.columns)
    fit = X @ b
    fit[~ok] = np.nan
    return b, fit, y - fit


def ols_summary(y, X, add_const=True):
    """Least-squares fit reported as a table: coefficient, standard error, t statistic
    and p value per regressor, plus R-squared and the number of observations used."""
    y = pd.Series(y)
    X = pd.DataFrame(X)
    if add_const:
        X = X.assign(const=1.0)
    ok = y.notna() & X.notna().all(axis=1)
    Xv, yv = X[ok].values, y[ok].values
    n, k = Xv.shape
    b, *_ = np.linalg.lstsq(Xv, yv, rcond=None)
    resid = yv - Xv @ b
    s2 = resid @ resid / (n - k)
    XtX_inv = np.linalg.pinv(Xv.T @ Xv)
    se = np.sqrt(np.diag(s2 * XtX_inv))
    t = b / se
    out = pd.DataFrame({"coef": b, "se": se, "t": t,
                        "p": 2 * (1 - stats.t.cdf(np.abs(t), n - k))}, index=X.columns)
    out.attrs["r2"] = 1 - resid.var() / yv.var()
    out.attrs["nobs"] = n
    print(f"n = {n},  R^2 = {out.attrs['r2']:.4f}")
    return out


def rolling_ols(y, X, window=104, add_const=True):
    """Refit `ols` on every trailing window of `window` rows.
    Returns a DataFrame of coefficients indexed by the end date of each window."""
    y = pd.Series(y)
    X = pd.DataFrame(X)
    if add_const:
        X = X.assign(const=1.0)
    idx, rows = [], []
    for i in range(window, len(y) + 1):
        ys, Xs = y.iloc[i - window:i], X.iloc[i - window:i]
        ok = ys.notna() & Xs.notna().all(axis=1)
        if ok.sum() < X.shape[1] + 5:
            continue
        b, *_ = np.linalg.lstsq(Xs[ok].values, ys[ok].values, rcond=None)
        idx.append(y.index[i - 1]); rows.append(b)
    return pd.DataFrame(rows, index=idx, columns=X.columns)


def residualize(y, X, add_const=True):
    """Return the part of y that is not explained by a least-squares fit on X."""
    return ols(y, X, add_const=add_const)[2]


# ---------------------------------------------------------------- dependence

def correlation_matrix(df, lag=0, method="pearson"):
    """Correlation between the columns of `df`. With lag = k, column i is compared
    against column j shifted forward by k rows, so entry (i, j) is
    corr(df[i] at t, df[j] at t-k)."""
    if lag == 0:
        return df.corr(method=method)
    out = pd.DataFrame(index=df.columns, columns=df.columns, dtype=float)
    for j in df.columns:
        sj = df[j].shift(lag)
        for i in df.columns:
            out.loc[i, j] = df[i].corr(sj, method=method)
    return out


def lagged_correlation(X, y, lags=range(1, 9), method="pearson"):
    """Correlation of y at time t against each column of X at times t-k, for k in `lags`.
    Returns a DataFrame with one row per column of X and one column per lag."""
    X = pd.DataFrame(X)
    y = pd.Series(y)
    return pd.DataFrame(
        {f"lag{k}": {c: y.corr(X[c].shift(k), method=method) for c in X.columns}
         for k in lags})


# ---------------------------------------------------------------- time series

def adf_test(x, maxlag=5, regression="c", verbose=True):
    """Augmented Dickey-Fuller test. The null hypothesis is that the series contains a
    unit root. Returns (statistic, p-value); more negative statistics favour rejection."""
    x = pd.Series(x).dropna()
    stat, p, *_ = adfuller(x, maxlag=maxlag, regression=regression, autolag=None)
    if verbose:
        print(f"ADF statistic = {stat:8.3f}    p = {p:.4f}    n = {len(x)}")
    return stat, p


def half_life(x):
    """Fit x[t] - x[t-1] = a + b * x[t-1] and report -log(2) / log(1 + b), the number of
    periods for a deviation to decay by half. Returns np.inf when b >= 0."""
    x = pd.Series(x).dropna()
    b, *_ = np.linalg.lstsq(
        np.column_stack([x.shift(1).values[1:], np.ones(len(x) - 1)]),
        x.diff().values[1:], rcond=None)
    rho = 1 + b[0]
    return np.inf if rho >= 1 or rho <= 0 else -np.log(2) / np.log(rho)


def ewma(x, halflife=10):
    """Exponentially weighted moving average."""
    return pd.Series(x).ewm(halflife=halflife, adjust=False).mean()


def zscore(x, window=52):
    """Standardise x against a trailing window of its own history."""
    x = pd.Series(x)
    r = x.rolling(window)
    return (x - r.mean()) / r.std()


def realized_vol(returns, window=4, periods_per_year=None):
    """Rolling standard deviation of a return series over `window` observations.
    If `periods_per_year` is given the result is scaled by its square root; the caller
    is responsible for passing a value that matches the frequency of `returns`."""
    s = pd.Series(returns).rolling(window).std()
    return s if periods_per_year is None else s * np.sqrt(periods_per_year)


# ---------------------------------------------------------------- other tools

def pca_factors(df, n_components=3, standardise=True):
    """Principal components of the columns of `df`. Returns (scores, loadings,
    explained_variance_ratio)."""
    Z = df.dropna()
    A = ((Z - Z.mean()) / Z.std()).values if standardise else (Z - Z.mean()).values
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    scores = pd.DataFrame(U[:, :n_components] * S[:n_components], index=Z.index,
                          columns=[f"PC{i+1}" for i in range(n_components)])
    loadings = pd.DataFrame(Vt[:n_components].T, index=Z.columns, columns=scores.columns)
    return scores, loadings, (S ** 2 / (S ** 2).sum())[:n_components]


def black_scholes(S, K, T, r, sigma, kind="call"):
    """Black-Scholes price. T is in years."""
    S, K, T, sigma = map(np.asarray, (S, K, T, sigma))
    with np.errstate(divide="ignore", invalid="ignore"):
        d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
    call = S * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    return call if kind == "call" else call - S + K * np.exp(-r * T)

def black_scholes_greeks(S, K, T, r, sigma):
    """Delta, gamma and vega of a Black-Scholes call. T is in years."""
    S, K, T, sigma = map(np.atleast_1d, map(np.asarray, (S, K, T, sigma)))
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    pdf = stats.norm.pdf(d1)
    return pd.DataFrame({"delta": stats.norm.cdf(d1),
                         "gamma": pdf / (S * sigma * np.sqrt(T)),
                         "vega":  S * pdf * np.sqrt(T)})


def implied_vol_from_price(price, S, K, T, r, kind="call", tol=1e-6, maxiter=100):
    """Invert `black_scholes` for sigma by bisection."""
    lo, hi = np.full_like(np.asarray(S, dtype=float), 1e-4), np.full_like(np.asarray(S, dtype=float), 5.0)
    for _ in range(maxiter):
        mid = 0.5 * (lo + hi)
        v = black_scholes(S, K, T, r, mid, kind)
        hi = np.where(v > price, mid, hi)
        lo = np.where(v > price, lo, mid)
        if np.nanmax(hi - lo) < tol:
            break
    return 0.5 * (lo + hi)


def seasonal_profile(x, by="dayofweek"):
    """Mean and count of x grouped by a calendar attribute of its index
    ('dayofweek', 'month', 'quarter', 'year')."""
    x = pd.Series(x).dropna()
    key = getattr(x.index, by)
    return pd.DataFrame({"mean": x.groupby(key).mean(),
                         "std":  x.groupby(key).std(),
                         "n":    x.groupby(key).size()})


def normality_test(x):
    """Skewness, excess kurtosis and the Jarque-Bera statistic of a series."""
    x = pd.Series(x).dropna()
    jb, p = stats.jarque_bera(x)[:2]
    return pd.Series({"skew": stats.skew(x), "excess_kurtosis": stats.kurtosis(x),
                      "jarque_bera": jb, "p": p, "n": len(x)})

---
## TODO A · Exploration

Work here. Nothing in this cell is graded numerically.

`matplotlib` is imported below — a plot is usually faster than a table for spotting
structure, and you are encouraged to use it.

Add as many cells as you like.

In [ ]:
# WRITE CODE HERE -- TODO A, your exploration.
#
# Starting points, if you want them:
#   panel.ret, panel.wret       returns, daily and weekly
#   panel.wrv                   weekly realized volatility
#   panel.wopt                  weekly option data
#   correlation_matrix(...), lagged_correlation(...), ols_summary(...)

import matplotlib.pyplot as plt

correlation_matrix(panel.wret).round(2)

---
## TODO B · Features

One task, three functions: `B-1` for T1, `B-2` for T2, `B-3` for T3. Each receives the
`Panel` and returns a DataFrame of predictors indexed by `panel.index`.

**Constraints — per function, not in total**

- at most **5 columns each** — this is enforced below
- row `t` may only use information dated on or before Friday `t`

The model fitted on top of your features is a fixed ordinary least-squares regression, so you
do not choose an estimator — you choose *what to give it*. Adding a column that carries no
information is not free: it costs you one of five slots and adds estimation noise.

The three functions are independent. A broken or empty one does not affect the others.

In [ ]:
# WRITE CODE HERE
def features_t1(panel):
    """TODO B-1 -- predictors for T1 (next week's return of e1)."""
    # ---- placeholder: shows the required shape, predicts nothing. Replace it. ----
    f = pd.DataFrame(index=panel.index)
    f["e1_last_week"] = panel.wret["e1"].shift(1)
    return f

In [ ]:
# WRITE CODE HERE
def features_t2(panel):
    """TODO B-2 -- predictors for T2 (next week's realized volatility of e2)."""
    # ---- placeholder: shows the required shape, predicts nothing. Replace it. ----
    f = pd.DataFrame(index=panel.index)
    f["idx_last_week"] = panel.wret["IDX"]
    return f

In [ ]:
# WRITE CODE HERE
def features_t3(panel):
    """TODO B-3 -- predictors for T3 (next week's return of the ETF)."""
    # ---- placeholder: shows the required shape, predicts nothing. Replace it. ----
    f = pd.DataFrame(index=panel.index)
    f["idx_last_week"] = panel.wret["IDX"]
    return f

---
## 5 · Fitting harness

Fixed. The estimator is ordinary least squares. It is fitted on the training rows and applied
unchanged to every other row — which, in your copy of the data, is none of them: your copy is
entirely training. When we score you, the same code fits on the same years and predicts the
years you do not have.

The score for one target is the **information coefficient** over the scored rows — the
correlation between your predictions $\hat y_t$ and the realised values $y_t$:

$$\mathrm{IC} \;=\; \operatorname{corr}\!\left(\hat y_t,\; y_t\right)$$

`evaluate` reports the in-sample IC. It is a diagnostic while you work — it is **not** your
score, and a high training IC is easy to obtain and easy to be misled by.

In [ ]:
# DO NOT TOUCH
def information_coefficient(pred, actual):
    """Correlation between predictions and realised values, over rows where both exist."""
    pred, actual = pd.Series(pred), pd.Series(actual)
    ok = pred.notna() & actual.notna()
    if ok.sum() < 3:
        return np.nan
    return float(np.corrcoef(pred[ok], actual[ok])[0, 1])


def _check_features(X, panel, name):
    if not isinstance(X, pd.DataFrame):
        raise TypeError(f"{name} must return a DataFrame, got {type(X).__name__}")
    if X.shape[1] > MAX_FEATURES:
        raise ValueError(f"{name} returned {X.shape[1]} columns; the limit is {MAX_FEATURES}")
    if X.shape[1] == 0:
        raise ValueError(f"{name} returned no columns")
    if not X.index.equals(panel.index):
        raise ValueError(f"{name} must be indexed by panel.index "
                         f"({len(panel.index)} rows), got {len(X.index)}")
    return X


def fit_predict(features_fn, target, panel, name="features"):
    """Fit OLS on the training rows and predict every row. Returns (predictions, coefficients)."""
    X = _check_features(features_fn(panel), panel, name)
    train, _ = split_masks(panel)
    D = X.assign(_const=1.0)
    ok = D.notna().all(axis=1) & target.notna()
    fit_rows = ok & train
    b, *_ = np.linalg.lstsq(D[fit_rows].values, target[fit_rows].values, rcond=None)
    b = pd.Series(b, index=D.columns)
    pred = (D @ b).where(D.notna().all(axis=1))
    return pred, b


def evaluate(panel=None, verbose=True):
    """Fit all three targets and report in-sample IC. Returns the prediction frame."""
    panel = build_panel() if panel is None else panel
    train, holdout = split_masks(panel)
    fns  = {"T1": features_t1, "T2": features_t2, "T3": features_t3}
    tgts = {"T1": make_target_t1(panel), "T2": make_target_t2(panel), "T3": make_target_t3(panel)}
    preds, rows = {}, []
    for k, fn in fns.items():
        p, b = fit_predict(fn, tgts[k], panel, name=fn.__name__)
        preds[k] = p
        rows.append({"target": k,
                     "n_features": len(b) - 1,
                     "IC_train": information_coefficient(p[train], tgts[k][train])})
    if verbose:
        print(pd.DataFrame(rows).to_string(index=False))
    return pd.DataFrame(preds)

In [ ]:
# DO NOT TOUCH
predictions = evaluate(panel)

---
## TODO C · Validation

You have no holdout. Every row you were given is a training row, so the IC printed above is
fitted in-sample and tells you almost nothing about how a feature will behave in a year you
have not seen.

A relationship that holds on average over fourteen years has not necessarily held *throughout*
those fourteen years, and a coefficient estimated once tells you nothing about its own
stability.

Work here in **code**: whatever evidence convinced you to keep a feature, and whatever
evidence made you drop another. The written explanation belongs in the write-up, not here.

In [ ]:
# WRITE CODE HERE -- TODO C, evidence that your chosen features are stable.
#
# Available, among others: rolling_ols(...), ols_summary(...), adf_test(...), half_life(...)

---
## Write-up

Submitted as a **separate document**, not in this notebook. **400 words maximum** across
questions 1-4.

1. **What is in the data?** Describe the structure you believe you found, and how you found it.
2. **Why do your features carry information about their target?**
3. **What ideas did you reject, and why?** Reasons other than a low IC.
4. **What would you do next, given more time?**

Optional, and not counted towards the word limit:

5. Did anything in the data look wrong, artificial, or otherwise strange to you? What did you
   do about it?

---
## 6 · Submission and scoring

Internal. Produces `submission.csv` and reports the holdout IC.


In [ ]:
# DO NOT TOUCH
def self_check(panel=None):
    """Confirm that the three feature functions run and leave usable values on every row."""
    panel = build_panel() if panel is None else panel
    for name, fn in [("features_t1", features_t1),
                     ("features_t2", features_t2),
                     ("features_t3", features_t3)]:
        try:
            X = _check_features(fn(panel), panel, name)
        except Exception as e:
            print(f"{name:12s} FAILED  -- {type(e).__name__}: {e}")
            continue
        usable = X.notna().all(axis=1).mean()
        notes = []
        if any(not np.issubdtype(X[c].dtype, np.number) for c in X.columns):
            notes.append("non-numeric column")
        if np.isinf(X.select_dtypes("number")).values.any():
            notes.append("inf values")
        const = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
        if const:
            notes.append(f"constant column {const}")
        if usable < 0.90:
            notes.append("rows with any NaN score as no prediction")
        print(f"{name:12s} OK  {X.shape[1]} feature(s)   usable rows {usable:6.1%}"
              + ("   ** " + "; ".join(notes) if notes else ""))



def make_submission(predictions, panel, path="submission.csv"):
    """Write the scored-period predictions to `path`."""
    _, holdout = split_masks(panel)
    out = predictions.loc[holdout, ["T1", "T2", "T3"]].copy()
    out.index.name = "Date"
    out.to_csv(path, float_format="%.10g")
    print(f"wrote {path}  ({len(out)} rows)")
    return out


def validate_submission(path="submission.csv", panel=None):
    """Check that a submission file has the expected shape, dates and dtypes."""
    panel = build_panel() if panel is None else panel
    _, holdout = split_masks(panel)
    want = panel.index[holdout]
    df = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")

    problems = []
    missing = [c for c in ("T1", "T2", "T3") if c not in df.columns]
    if missing:
        problems.append(f"missing columns: {missing}")
    if not df.index.equals(want):
        problems.append(f"dates do not match the scored grid "
                        f"(expected {len(want)} rows, found {len(df)})")
    for c in df.columns.intersection(["T1", "T2", "T3"]):
        if not np.issubdtype(df[c].dtype, np.number):
            problems.append(f"{c} is not numeric")
        elif df[c].notna().sum() == 0:
            problems.append(f"{c} is entirely empty")
        elif df[c].std() == 0:
            problems.append(f"{c} is constant -- IC is undefined")
        else:
            n_na = int(df[c].isna().sum())
            if n_na:
                problems.append(f"{c} has {n_na} missing values (they score as no prediction)")

    if problems:
        print("PROBLEMS")
        for p in problems:
            print("  -", p)
    else:
        print(f"OK -- {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")
    return df


def score(predictions, panel):
    """Holdout IC per target. Internal use only."""
    _, holdout = split_masks(panel)
    tg = {"T1": make_target_t1(panel), "T2": make_target_t2(panel), "T3": make_target_t3(panel)}
    for k in ("T1", "T2", "T3"):
        ic = information_coefficient(predictions[k][holdout], tg[k][holdout])
        print(f"  {k}   IC_holdout = {ic:+.4f}")


self_check(panel)
print()
_ = make_submission(predictions, panel)
_ = validate_submission(panel=panel)
print()
score(predictions, panel)
